# Ejecución completa del Tema 55 (Alejandro Soto Antezana, 2024200531K)
Ejecute las celdas **en orden**, de arriba hacia abajo, con el botón ▶ de cada una.

## Celda 1. Conectar Google Drive y entrar a la carpeta del proyecto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/3_Base_de_datos_y_codigo_Alejandro_Soto'
%cd "$CARPETA"
!ls

## Celda 2. Instalar librerías

In [ ]:
!pip install -q -r requirements.txt

## Celda 3. Crear el archivo .env (solo en Colab; NUNCA se sube a GitHub)

In [ ]:
%%writefile .env
SEC_USER_AGENT="Alejandro Soto Antezana e_2024200531K@uncp.edu.pe"

## Celda 4. Vía 1: API (BCRP + Yahoo Finance)

In [ ]:
!python codigo/01_extraccion_api.py

## Celda 5. Vía 2: descarga programática (SEC + SPDR)

In [ ]:
!python codigo/02_scraping_web.py

## Celda 6. Limpieza, unión, hash y tabla de cotejo

In [ ]:
!python codigo/03_limpieza_datos.py

## Celda 7. Análisis y Monte Carlo (tablas y figuras en salidas/)

In [ ]:
!python codigo/04_analisis.py

## Celda 8. requirements.txt con versiones exactas y README rellenado por código
Escriba antes el enlace de su repositorio en `ENLACE_REPO`.

In [ ]:
import sys, hashlib, subprocess, pandas as pd
from importlib.metadata import version
ENLACE_REPO = "https://github.com/USUARIO/REPOSITORIO"   # <- cambie esto

libs = ["requests", "pandas", "numpy", "yfinance", "pdfplumber", "statsmodels", "scipy",
        "matplotlib", "openpyxl", "python-dotenv"]
lineas = [f"{l}=={version(l)}" for l in libs]
open("requirements.txt", "w").write("\n".join(lineas) + "\n")

proc = "datos_procesados/datos_procesados_2024200531K.csv"
base = pd.read_csv(proc)
sha = hashlib.sha256(open(proc, "rb").read()).hexdigest()
readme = open("README.md", encoding="utf-8").read()
for clave, valor in {"{{ENLACE_REPO}}": ENLACE_REPO, "{{PRIMERA_FECHA}}": base["fecha"].iloc[0],
                     "{{ULTIMA_FECHA}}": base["fecha"].iloc[-1], "{{FILAS}}": f"{len(base):,}",
                     "{{VERSION_PYTHON}}": sys.version.split()[0], "{{LIBRERIAS}}": ", ".join(lineas),
                     "{{HASH}}": sha}.items():
    readme = readme.replace(clave, str(valor))
open("README.md", "w", encoding="utf-8").write(readme)
print(open("requirements.txt").read()); print("SHA-256:", sha)

## Celda 9. Empaquetar lo que debo enviar a Claude para revisar y redactar el artículo

In [ ]:
import shutil, os
os.makedirs("/content/para_claude", exist_ok=True)
for f in ["log_ejecucion.txt", "README.md", "datos_crudos/datos_crudos_2024200531K_sec_trs_parametros.csv",
          "datos_procesados/parametros_proyecto_2024200531K.csv"]:
    if os.path.exists(f): shutil.copy(f, "/content/para_claude/")
shutil.copytree("salidas", "/content/para_claude/salidas", dirs_exist_ok=True)
shutil.make_archive("/content/resultados_para_claude", "zip", "/content/para_claude")
from google.colab import files
files.download("/content/resultados_para_claude.zip")